In [0]:
MY_ID = "preetam"
VOL = f"/Volumes/workspace/capstone_{MY_ID}/raw"

spark.sql(f"CREATE CATALOG IF NOT EXISTS workspace")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS workspace.capstone_{MY_ID}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS workspace.capstone_{MY_ID}.raw")

DataFrame[]

In [0]:
# STEP 2 — Bronze (land it, change nothing)

import pyspark.sql.functions as F

def bronze(name):
    df = spark.read.csv(f'{VOL}/raw/{name}', header=True, inferSchema=False)
    return (df.withColumn("_source_file", F.col("_metadata.file_path"))
              .withColumn("_ingested_at", F.current_timestamp())
              .withColumn("_row_hash", F.sha2(F.concat_ws("||", *df.columns), 256)))

bronze_marks    = bronze("marks")
bronze_students = bronze("students")
bronze_subjects = bronze("subjects")

bronze_marks.write.mode("overwrite").saveAsTable(f"workspace.capstone_{MY_ID}.bronze_marks")
bronze_students.write.mode("overwrite").saveAsTable(f"workspace.capstone_{MY_ID}.bronze_students")
bronze_subjects.write.mode("overwrite").saveAsTable(f"workspace.capstone_{MY_ID}.bronze_subjects")

print(bronze_marks.count(), bronze_students.count(), bronze_subjects.count())

22200 1200 6
